## Questão 1


In [15]:
import numpy as np
import os
import soundfile as sf
from scipy.signal import lfilter

# =====================================================
# 1. FUNÇÕES AUXILIARES
# =====================================================

def apply_delay(x, m):
    """ Aplica um atraso puro de m amostras (Para o Campo Direto) """
    y = np.zeros(len(x) + m)
    y[m:] = x
    return y

def schroeder_reverb(x, Fs, RT60):
    """ Reverberador de Schroeder (R(z)) """
    # Se RT60 for 0, não existe cauda de reverb
    if RT60 <= 0:
        return np.zeros_like(x)

    # Atrasos e Ganhos (Schroeder standard)
    comb_delays = np.array([1557, 1617, 1491, 1422])
    ap_delays   = np.array([225, 556])

    # Cálculo do ganho de feedback g = 10^(-3 * delay / RT60)
    g_comb = 10 ** (-3 * comb_delays / (RT60 * Fs))

    y = np.zeros_like(x, dtype=float)

    # 1. Filtros Comb em Paralelo
    for d, g in zip(comb_delays, g_comb):
        # A lógica lfilter para comb feedback: y[n] = x[n] - g*y[n-d]
        # Numerador b=[1], Denominador a=[1, 0...0, -g]
        b = np.zeros(d + 1); b[0] = 1
        a = np.zeros(d + 1); a[0] = 1; a[-1] = -g # Feedback
        y += lfilter(b, a, x)

    # 2. Filtros All-Pass em Série
    g_ap = 0.7
    for d in ap_delays:
        b = np.zeros(d + 1); a = np.zeros(d + 1)
        b[0] = -g_ap; b[-1] = 1
        a[0] = 1;     a[-1] = g_ap
        y = lfilter(b, a, y)

    return y

# =====================================================
# 2. CONFIGURAÇÃO FÍSICA
# =====================================================
Fs = 44100
c  = 343

d1, d2, d3 = 1.5, 2.2, 3.1
a1, a2, a3 = 1.0/d1, 1.0/d2, 1.0/d3

m1 = int(d1 / c * Fs)
m2 = int(d2 / c * Fs)
m3 = int(d3 / c * Fs)

# =====================================================
# 3. LEITURA E SOMA (CORRIGIDA)
# =====================================================
folder = "FragileThoughts/"

# Tenta listar ficheiros
try:
    tracks = [f for f in os.listdir(folder) if f.lower().endswith(".wav")]
except FileNotFoundError:
    tracks = []

if not tracks:
    print("AVISO: Pasta não encontrada. A gerar ruído de teste.")
    x = np.random.normal(0, 0.1, Fs*2) # 2 segundos de ruído
else:
    print(f"A carregar {len(tracks)} pistas...")
    
    # 1. Ler a primeira pista corretamente
    path0 = os.path.join(folder, tracks[0])
    x_soma, Fs = sf.read(path0)
    if x_soma.ndim > 1: x_soma = x_soma[:, 0] # Mono

    # 2. Somar as restantes (começar do índice 1 para não duplicar a primeira!)
    for t in tracks[1:]:
        path = os.path.join(folder, t)
        sig, _ = sf.read(path)
        if sig.ndim > 1: sig = sig[:, 0]

        # Ajuste de tamanho
        min_len = min(len(x_soma), len(sig))
        x_soma = x_soma[:min_len] # Corta o excesso
        x_soma += sig[:min_len]   # Soma
        print(f" -> + {t}")

    # 3. Normalizar Input
    peak = np.max(np.abs(x_soma))
    if peak > 1.0:
        x_soma /= peak
        print(" -> Input normalizado (evitou clipping na soma).")
    
    x = x_soma

print("Sinal de entrada pronto.")

# =====================================================
# QUESTÃO 1 (a) - CAMPO DIRETO
# =====================================================
# Atraso simples + Atenuação
campo_direto = apply_delay(x, m1) * a1
sf.write("Q1_a_campo_direto.wav", campo_direto, Fs)
print("Q1a gerado.")

# =====================================================
# QUESTÃO 1 (b) - PRIMEIRAS REFLEXÕES (MÉTODO FIR)
# =====================================================
# CORREÇÃO: Usar lfilter em vez de somar arrays manualmente.
# H(z) = a2 * z^-m2 + a3 * z^-m3

max_delay = max(m2, m3)
b_fir = np.zeros(max_delay + 1) # Numerador do filtro FIR

# Colocar os ganhos nas posições dos atrasos (Impulse Response da sala)
b_fir[m2] = a2
b_fir[m3] = a3
a_fir = np.array([1]) # Denominador é 1 (FIR)

# Filtragem
campo_reflexoes = lfilter(b_fir, a_fir, x)
sf.write("Q1_b_primeiras_reflexoes.wav", campo_reflexoes, Fs)
print("Q1b gerado (via Filtro FIR).")

# =====================================================
# QUESTÃO 1 (c) e (d) - REVERB E TOTAL
# =====================================================
RT60_values = [0, 0.5, 2, 10]

for RT in RT60_values:
    # --- Q1c: Campo Reverberante Isolado ---
    # Gerar reverb
    rv = schroeder_reverb(x, Fs, RT)
    
    # Gravar Q1c (Normalizado para não ser silêncio absoluto ou estoiro)
    rv_to_save = rv.copy()
    peak_rv = np.max(np.abs(rv_to_save))
    if peak_rv > 0: 
        rv_to_save /= peak_rv # Normaliza para 0dB para audição clara
    
    sf.write(f"Q1_c_reverb_RT{RT}.wav", rv_to_save, Fs)
    
    # --- Q1d: Campo Total ---
    # Total = Direto + Reflexões + (Wet * Reverb)
    
    # 1. Definir ganho do Reverb (Wet)
    # Se RT=0, wet=0. Se RT>0, wet=0.05 (para não abafar o som direto)
    wet = 0.05 if RT > 0 else 0
    
    # 2. Igualar tamanhos (Padding)
    L_total = max(len(campo_direto), len(campo_reflexoes), len(rv))
    
    # Função rápida para padding
    def pad(v, size): return np.pad(v, (0, size - len(v)))
    
    # Soma
    sinal_total = pad(campo_direto, L_total) + \
                  pad(campo_reflexoes, L_total) + \
                  (pad(rv, L_total) * wet)
                  
    # 3. Normalização de Segurança
    peak_tot = np.max(np.abs(sinal_total))
    if peak_tot > 1.0:
        sinal_total /= peak_tot
        print(f" -> RT{RT}: Normalizado (Pico: {peak_tot:.2f})")
        
    sf.write(f"Q1_d_total_RT{RT}.wav", sinal_total, Fs)

print("Processo concluído.")

A carregar 7 pistas...
 -> + 02_Snare.wav
 -> + 03_Overhead.wav
 -> + 04_BassDI.wav
 -> + 05_Gtr.wav
 -> + 06_Percussion.wav
 -> + 07_LeadVox.wav
 -> Input normalizado (evitou clipping na soma).
Sinal de entrada pronto.
Q1a gerado.
Q1b gerado (via Filtro FIR).
 -> RT0.5: Normalizado (Pico: 1.05)
 -> RT2: Normalizado (Pico: 1.28)


KeyboardInterrupt: 

## Questão 2

In [ ]:
import numpy as np
import soundfile as sf
import os
from scipy.signal import lfilter

# =====================================================
# 1. CONFIGURAÇÃO
# =====================================================
# Ajusta este caminho conforme a tua estrutura real
PASTA_HRTF = "HRTF/compact/elev0/" 

# =====================================================
# 2. FUNÇÕES AUXILIARES
# =====================================================

def apply_delay(x, m):
    """Aplica atraso de m amostras"""
    if m == 0: return x
    y = np.zeros(len(x) + m)
    y[m:] = x
    return y

def schroeder_reverb(x, Fs, RT60):
    """Gera apenas a componente reverberante (Wet)"""
    # Se RT60 for 0, não há reverberação (retorna silêncio)
    if RT60 <= 0: return np.zeros_like(x)

    comb_delays = np.array([1557, 1617, 1491, 1422])
    ap_delays   = np.array([225, 556])
    
    # Cálculo do ganho de feedback (Schroeder)
    g_comb = 10 ** (-3 * comb_delays / (RT60 * Fs))

    y = np.zeros_like(x, dtype=float)
    
    # 1. Filtros Comb em Paralelo
    for d, g in zip(comb_delays, g_comb):
        # Implementação eficiente: y[n] = x[n] - g*y[n-d] (Feedback Comb)
        # Nota: A tua implementação FIR/IIR estava correta, mantive a lógica lfilter
        b = np.zeros(d + 1); b[0] = 1
        a = np.zeros(d + 1); a[0] = 1; a[-1] = -g # Feedback negativo é comum, ou positivo
        # Para Schroeder original geralmente usa-se ganho positivo no loop, mas varia.
        # Vamos manter a tua estrutura que é estável.
        y += lfilter(b, a, x)

    # 2. Filtros All-Pass em Série
    g_ap = 0.7
    for d in ap_delays:
        b = np.zeros(d + 1); a = np.zeros(d + 1)
        b[0] = -g_ap; b[-1] = 1
        a[0] = 1;     a[-1] = g_ap
        y = lfilter(b, a, y)
        
    return y

def binaural_hrtf_dat(x, angle_deg):
    """
    Lê HRTF .dat e aplica simetria para bases 0-180º.
    """
    # 1. Normalizar ângulo para 0-359 (trata negativos automaticamente)
    angle_clean = int(5 * round(angle_deg / 5)) % 360
    
    # 2. Lógica de Simetria (MIT KEMAR Compact)
    # A base tem ficheiros de 0 a 180 (lado esquerdo).
    # Se o ângulo for > 180 (lado direito), espelhamos e trocamos canais.
    swap_channels = False
    
    if angle_clean > 180:
        angle_clean = 360 - angle_clean
        swap_channels = True
        
    filename = f"H0e{angle_clean:03d}a.dat"
    filepath = os.path.join(PASTA_HRTF, filename)

    # 3. Fallback se ficheiro não existir
    if not os.path.exists(filepath):
        print(f"[AVISO] HRTF em falta: {filename}. A usar Mono.")
        return x, x 
        
    try:
        # Ler Raw Big-Endian 16-bit (>i2)
        raw_data = np.fromfile(filepath, dtype='>i2')
        # Normalizar para float -1.0 a 1.0
        hrtf = raw_data.astype(float) / 32768.0
        hrtf = hrtf.reshape(-1, 2)
        
        ir_L = hrtf[:, 0]
        ir_R = hrtf[:, 1]
        
        # 4. Trocar canais se for do lado direito
        if swap_channels:
            ir_L, ir_R = ir_R, ir_L
            
        # 5. Convolução
        xL = np.convolve(x, ir_L, mode='full')
        xR = np.convolve(x, ir_R, mode='full')
        
        return xL, xR
        
    except Exception as e:
        print(f"Erro no ficheiro {filename}: {e}")
        return x, x

# =====================================================
# 3. LEITURA DE SINAL (MISTURA)
# =====================================================
folder = "FragileThoughts/"
# Verifica se a pasta existe, senão cria dummy
if not os.path.exists(folder):
    print("Pasta não encontrada. A usar ruído branco.")
    Fs = 48000
    x = np.random.normal(0, 0.5, 48000)
else:
    files = [f for f in os.listdir(folder) if f.endswith(".wav")]
    if files:
        print(f"A processar {len(files)} ficheiros...")
        path0 = os.path.join(folder, files[0])
        x_soma, Fs = sf.read(path0)
        if x_soma.ndim > 1: x_soma = x_soma[:, 0]
        
        for filename in files[1:]:
            path = os.path.join(folder, filename)
            sig, _ = sf.read(path)
            if sig.ndim > 1: sig = sig[:, 0]
            
            min_len = min(len(x_soma), len(sig))
            x_soma = x_soma[:min_len] + sig[:min_len]
            print(f" -> Somado: {filename}")
        
        # Normalização de entrada (Segurança)
        x = x_soma
        peak = np.max(np.abs(x))
        if peak > 0.9:
            x = x / peak * 0.9
            print(" -> Input normalizado para 0.9")
    else:
        Fs = 48000
        x = np.zeros(Fs)

# =====================================================
# 4. EXECUÇÃO DOS EXERCÍCIOS
# =====================================================

# Parâmetros Geométricos
c = 343
d1, d2, d3 = 1.5, 2.2, 3.1
m1 = int(d1/c*Fs)
m2 = int(d2/c*Fs)
m3 = int(d3/c*Fs)

# Ângulos (Fonte Frente, Refletor Esq, Refletor Dir)
ang_direto = 30    # Frente
ang_ref1   = -60   # 45º Esquerda
ang_ref2   = 110 # 45º Direita (vai ser tratado como 315º)

# --- ALÍNEA A: Efeito da Absorção (Alpha) ---
print("\n--- A processar Q2a (Absorção) ---")
alphas = [0, 0.5, 0.8, 1]

# Variável para guardar o som "seco" (sem reverb) para usar na alínea B
# Vamos guardar o caso intermédio (ex: alpha 0.5) ou o alpha=0.
sinal_seco_referencia_L = None
sinal_seco_referencia_R = None
ALPHA_REFERENCIA = 0.5 

for alpha in alphas:
    # 1. CÁLCULO DOS GANHOS (CORREÇÃO FÍSICA AQUI)
    # Som Direto: atenua só com distância (1/d)
    g1 = 1.0 / d1 
    
    # Reflexões: atenuam com distância E com absorção da parede
    # Coef Reflexão (R) = sqrt(1 - alpha)
    reflexao_coef = np.sqrt(1.0 - alpha)
    
    g2 = reflexao_coef / d2
    g3 = reflexao_coef / d3
    
    # 2. Criar os atrasos com os ganhos aplicados
    s_d  = apply_delay(x, m1) * g1
    s_r1 = apply_delay(x, m2) * g2
    s_r2 = apply_delay(x, m3) * g3
    
    # 3. Espacialização (HRTF)
    Ld, Rd   = binaural_hrtf_dat(s_d, ang_direto)
    Lr1, Rr1 = binaural_hrtf_dat(s_r1, ang_ref1)
    Lr2, Rr2 = binaural_hrtf_dat(s_r2, ang_ref2)
    
    # 4. Soma (Padding para evitar erros de tamanho)
    maxlen = max(len(Ld), len(Lr1), len(Lr2))
    def pad(v, l): return np.pad(v, (0, l - len(v)))
    
    mix_L = pad(Ld, maxlen) + pad(Lr1, maxlen) + pad(Lr2, maxlen)
    mix_R = pad(Rd, maxlen) + pad(Rr1, maxlen) + pad(Rr2, maxlen)
    
    # Guardar referência para a alínea B
    if alpha == ALPHA_REFERENCIA:
        sinal_seco_referencia_L = mix_L
        sinal_seco_referencia_R = mix_R

    # 5. Gravar
    # IMPORTANTE: Não normalizar cada um ao máximo individualmente, 
    # senão perdemos a noção de que alpha=1 é mais silencioso que alpha=0.
    # Vamos usar um fator de escala global fixo (ajusta conforme necessário)
    out = np.column_stack((mix_L, mix_R))
    
    # Verifica clipping apenas
    p = np.max(np.abs(out))
    if p > 1.0:
        out /= p
        print(f" -> Aviso: Alpha {alpha} clipou e foi normalizado.")
    
    sf.write(f"Q2a_Final_Alpha{alpha}.wav", out, Fs)
    print(f" -> Gerado Alpha {alpha}")


# --- ALÍNEA B: Adicionar Reverb (RT60 variável) ---
print("\n--- A processar Q2b (Reverb) ---")
# Vamos usar o sinal seco gerado com Alpha=0.5 como base
if sinal_seco_referencia_L is None:
    print("Erro: Referência não gerada na Q2a. Verifica o ALPHA_REFERENCIA.")
    exit()

RT60_vals = [0, 0.5, 2, 10]

for RT in RT60_vals:
    # 1. Gerar Reverb (Mono)
    rev_mono = schroeder_reverb(x, Fs, RT)
    
    # 2. Espacializar Reverb
    # O reverb numa sala é difuso (vem de todo o lado).
    # Uma aproximação simples é passar pela HRTF 0º ou somar direto com baixo ganho.
    # Vamos passar pela HRTF 0º para manter consistência espectral.
    rL, rR = binaural_hrtf_dat(rev_mono, 0)
    
    # 3. Mistura Wet/Dry
    wet_gain = 0.05 # O reverb deve ser subtil em relação ao som direto
    
    if RT == 0: 
        wet_gain = 0 # Garante silêncio absoluto no reverb
    
    # Soma com o sinal seco (Dry + Early Reflections)
    maxlen = max(len(sinal_seco_referencia_L), len(rL))
    
    final_L = pad(sinal_seco_referencia_L, maxlen) + (pad(rL, maxlen) * wet_gain)
    final_R = pad(sinal_seco_referencia_R, maxlen) + (pad(rR, maxlen) * wet_gain)
    
    # 4. Gravar
    out = np.column_stack((final_L, final_R))
    
    # Normalização de segurança
    p = np.max(np.abs(out))
    if p > 1.0: out /= p
        
    sf.write(f"Q2b_Final_RT{RT}.wav", out, Fs)
    print(f" -> Gerado RT {RT}s (Base Alpha {ALPHA_REFERENCIA})")

print("\nConcluído com sucesso.")

A processar 7 ficheiros...
 -> Somado: 02_Snare.wav
 -> Somado: 03_Overhead.wav
 -> Somado: 04_BassDI.wav
 -> Somado: 05_Gtr.wav
 -> Somado: 06_Percussion.wav
 -> Somado: 07_LeadVox.wav
 -> Input normalizado para 0.9

--- A processar Q2a (Absorção) ---
 -> Gerado Alpha 0
 -> Gerado Alpha 0.5
 -> Gerado Alpha 0.8
 -> Gerado Alpha 1

--- A processar Q2b (Reverb) ---
 -> Gerado RT 0s (Base Alpha 0.5)
 -> Gerado RT 0.5s (Base Alpha 0.5)
 -> Gerado RT 2s (Base Alpha 0.5)
 -> Gerado RT 10s (Base Alpha 0.5)

Concluído com sucesso.


## Questão 3

In [ ]:
import numpy as np
import soundfile as sf

# =====================================================
# QUESTÃO 3
# Reprodução Ambisonic em sistema multicanal (8.0)
# =====================================================

print("A iniciar processamento Ambisonic...")

# 1. RECUPERAR SINAIS E DADOS DAS QUESTÕES ANTERIORES
# -----------------------------------------------------
# (Assume-se que as variáveis campo_direto, reflexao_1, reflexao_2,
#  e os ângulos ang_direto, ang_ref1, ang_ref2 já existem na memória
#  após correr a Q1 e Q2.).

# Assegurar que os ângulos estão em radianos
theta_d  = np.deg2rad(ang_direto) # 30º
theta_r1 = np.deg2rad(ang_ref1)   # -60º
theta_r2 = np.deg2rad(ang_ref2)   # 110º

# Igualar o comprimento dos vetores (Padding) para poder somar
L_max = max(len(campo_direto), len(reflexao_1), len(reflexao_2))

s_d  = np.pad(campo_direto, (0, L_max - len(campo_direto)))
s_r1 = np.pad(reflexao_1,   (0, L_max - len(reflexao_1)))
s_r2 = np.pad(reflexao_2,   (0, L_max - len(reflexao_2)))


# 2. CODIFICAÇÃO (B-FORMAT - 1ª ORDEM)
# -----------------------------------------------------
# W = S / sqrt(2)
# X = S * cos(theta)
# Y = S * sin(theta)

# Calculamos W, X, Y totais (soma das 3 fontes sonoras codificadas)
W = (s_d + s_r1 + s_r2) / np.sqrt(2)

X = (s_d * np.cos(theta_d)) + \
    (s_r1 * np.cos(theta_r1)) + \
    (s_r2 * np.cos(theta_r2))

Y = (s_d * np.sin(theta_d)) + \
    (s_r1 * np.sin(theta_r1)) + \
    (s_r2 * np.sin(theta_r2))

# Guardar o B-Format (opcional, para análise)
sf.write("Q3_Ambisonic_BFormat_WXY.wav", 
         np.column_stack((W, X, Y)), Fs)


# 3. GEOMETRIA DOS ALTIFALANTES
# -----------------------------------------------------
# 8 altifalantes em círculo, igualmente espaçados (45º entre cada)
# 0º geralmente é Frente. Sentido anti-horário.
num_speakers = 8
spk_angles_deg = np.linspace(0, 360, num_speakers, endpoint=False)
spk_angles_rad = np.deg2rad(spk_angles_deg)


# 4. DESCODIFICAÇÃO (VIRTUAL MICROPHONES)
# -----------------------------------------------------
# Slide 40: Microfone virtual com p=0.5 (Cardioide)
# Sinal = p * Omni + (1-p) * Figure8_na_direcao_do_altifalante
p = 0.5 

speaker_signals = []

for phi in spk_angles_rad:
    # Componente Omnidirecional (recuperada de W)
    omni_comp = np.sqrt(2) * W
    
    # Componente Figura-8 projetada na direção do altifalante (phi)
    fig8_comp = X * np.cos(phi) + Y * np.sin(phi)
    
    # Soma ponderada pelo fator de diretividade p
    sig = p * omni_comp + (1 - p) * fig8_comp
    
    speaker_signals.append(sig)

# Converter lista para matriz (amostras x 8 canais)
multichannel_out = np.column_stack(speaker_signals)

# Normalização de segurança (para evitar clipping no wav multicanal)
peak = np.max(np.abs(multichannel_out))
if peak > 1.0:
    multichannel_out /= peak
    print(f"Sinal normalizado (Peak attenuation: {1/peak:.2f})")

# 5. ESCRITA DO FICHEIRO FINAL
# -----------------------------------------------------
output_filename = "Q3_Ambisonic_Decoded_8ch.wav"
sf.write(output_filename, multichannel_out, Fs)

print(f"Questão 3 concluída.")
print(f"Ficheiro gerado: {output_filename} ({num_speakers} canais).")
print(f"Ficheiro B-Format gerado: Q3_Ambisonic_BFormat_WXY.wav")